# PyTorch Foundations 01 - A model doesn't learn just because we call it

**Guiding question:** What actually changes a bad guess into a better one?

You already saw hidden features and backpropagation in the ML foundations.
Now you want to make those ideas run in PyTorch without hiding everything inside
one mysterious `train()` call.

We could start with a word guesser, like the later language-model lessons.
But then we'd also need word IDs, embedding rows, and several possible answers.
Let's keep just one number going in and one number coming out, so you can see
exactly which part makes a guess, which part checks it, and which part changes it.
The word guessers will use this same practice routine later.

Our example is a cheap sensor with a crooked response: its raw reading isn't
the calibrated value we want. We give the model pairs of readings and reference
values, then ask it to learn the correction.

![A sensor model guesses, checks its error, traces gradients, and adjusts its weights before guessing again.](images/01-01-guess-check-trace-adjust.png)
*Read the stages from left to right. Only the last stage changes the weights.
The illustration explains the jobs; the code below measures what each one does.*

## What you'll build

| The complaint | What we'll try | What you'll check |
|---|---|---|
| The sensor readings need a correction | Make input/reference pairs | One row per reading, one value per row |
| A straight response can't follow a curve | Twelve hidden values and a bend | The input and output shapes still match |
| The model runs, but its guesses are poor | Compare guesses with the reference | A visible gap and one loss number |
| An error score doesn't tell us what changed | Separate backward from the optimizer | Weights before backward, after backward, and after step |
| One correction isn't enough | Repeat the same practice loop | Recorded loss and a before/after curve |
| We only want a prediction now | Switch off learning's bookkeeping | New predictions without a gradient graph |

**What you need:** the previous ML foundations and the chapter's setup environment.
**Where we stop:** one small regression model and its training loop. Batches,
validation, saved training state, and GPU training tricks come later.

### Before the first experiment

PyTorch gives us the number containers and automatic gradient calculation.
`nn` contains building blocks for the model; Matplotlib draws the evidence.
We fix a random seed so the starting weights are repeatable. That doesn't make
them useful yet. CPU is plenty for this lesson; a GPU also works if one is available.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; device={device}")

# Find this lesson even when VS Code starts the kernel at the repository root.
LESSON_DIR = next(
    folder for folder in (
        Path(globals().get("__vsc_ipynb_file__", "")).resolve().parent,
        Path.cwd(),
        Path.cwd() / "learning" / "02-pytorch-foundations" / "01-deep-learning-map",
    )
    if (folder / "01-deep-learning-map.ipynb").is_file()
)

## 1 - Give the guesser something to learn from

A raw sensor reading isn't necessarily the calibrated value you want. Put
each reading beside a trusted reference: the reading is the input; the
reference is the checking answer, or **target**. Predicting a number is
**regression**.

Here we invent 160 pairs from a curved rule. To see the bend before worrying
about a model, check a few easy points on that rule:

| Raw reading | Reference value |
|---:|---:|
| -2 | -3.8 |
| 0 | 0.1 |
| 2 | 4.0 |

These are arithmetic landmarks, not three selected training rows; the
160-point grid includes the endpoints but not exactly zero. The model gets
input/target pairs, not the rule that made them.

`linspace` makes 160 readings in a flat list. `unsqueeze(1)` puts them into
one column without adding measurements:

```text
x [160, 1]: 160 examples, one raw reading each
y [160, 1]: the same examples, one reference value each
```

> **Visual to add — The crooked sensor ruler**
> Show the raw readings -2, 0, 2 on a straight input ruler, connected to
> reference heights -3.8, 0.1, 4.0 on a curved response. Keep one paired
> reading/reference card beside the rulers; don't suggest that zero is a
> row in the 160-point grid. Contrast a straight correction with the curved
> reference. Suggested caption: "The input is one reading; the target tells
> us which correction that reading needs."

**Predict:** Will adding the feature-column axis change the values or just
their grouping? The next printout shows the shapes and actual practice pairs.


In [ ]:
# [160] -> [160, 1]: give each reading its own row with one feature column.
x = torch.linspace(-2.0, 2.0, 160).unsqueeze(1).to(device)
# Our made-up reference curve keeps the same rows: one target per reading.
y = 0.55 * x**3 - 0.25 * x + 0.10

print(f"x: {tuple(x.shape)} = (examples, raw-reading columns)")
print(f"y: {tuple(y.shape)} = (examples, reference-value columns)")
print("A few practice pairs:")
for index in (0, 40, 80, 120, 159):
    print(f"  reading {x[index].item():+.2f} -> reference {y[index].item():+.3f}")

assert x.shape == y.shape == (160, 1)

## 2 - Build an adjustable guesser

A single straight-line correction could tilt and shift its guesses.
Our reference curve bends, so that isn't enough.

Give the model twelve different adjustable straight-line calculations of the
same reading. Then bend each result with **`tanh`**, which smoothly squeezes it
between -1 and 1. Finally, combine those twelve bent responses into one guess.
Different combinations can follow a curve.

Without that bend, stacking two straight-line calculations still gives you
one straight line. More layers alone wouldn't fix our problem.

![One reading becomes twelve hidden values and then one prediction. The batch keeps N examples throughout.](images/01-02-one-reading-twelve-features.png)
*The top row follows one reading. The bottom row names the batch shapes.
Twelve features means twelve values per reading, not twelve copies of the dataset.*

### What do these twelve values mean?

We don't name them "temperature" or "sensor age". They are **hidden features**:
intermediate numbers that training can make useful. This is the same habit
you'll use with learned word rows later: useful numbers needn't have names
we chose by hand.

An **`nn.Module`** keeps the adjustable numbers, called **parameters**, together
with the instructions for using them. `__init__` creates the layers.
`forward` describes one trip through them. Calling `model(x)` runs that trip;
it doesn't train anything by itself.

**Predict:** When one reading becomes twelve features, do we suddenly have
twelve guesses? No. The last layer turns those features back into one guess.

We'll also check the bend at three equally spaced readings. A straight line's
middle answer sits halfway between the outer answers. Running the same layers
with and without `tanh` lets us measure whether that changes.

In [ ]:
class TinyRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        # One reading -> twelve adjustable responses to that reading.
        self.hidden = nn.Linear(1, 12)
        # Combine those twelve features into one calibrated-value guess.
        self.output = nn.Linear(12, 1)

    def forward(self, inputs):
        # [N, 1] -> [N, 12]: more features in each row, not more examples.
        # tanh bends the features but keeps the [N, 12] shape.
        hidden = torch.tanh(self.hidden(inputs))
        # [N, 12] -> [N, 1]: bring each row back to one prediction.
        return self.output(hidden)


model = TinyRegressor().to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"The model has {parameter_count} adjustable numbers.")
# Hidden layer: 12 weights + 12 biases. Output: 12 weights + 1 bias.
assert parameter_count == 37

# At three equally spaced readings, a straight line's middle answer is halfway
# between the outer answers. Reuse our layers with/without tanh to check the bend.
bend_probe = torch.tensor([[-1.0], [0.0], [1.0]], device=device)
with torch.inference_mode():
    straight_answers = model.output(model.hidden(bend_probe))
    bent_answers = model(bend_probe)
    straight_gap = (straight_answers[1] - (straight_answers[0] + straight_answers[2]) / 2).abs().item()
    bent_gap = (bent_answers[1] - (bent_answers[0] + bent_answers[2]) / 2).abs().item()
print(f"Middle answer's gap from a straight line: without tanh {straight_gap:.6f}; with tanh {bent_gap:.6f}")
assert straight_gap < 1e-6
assert bent_gap > 1e-4

### Read the class as a short trip

The class creates two layers once and reuses them for every reading.
For hidden feature number 1, the first layer multiplies the reading by its
own weight and adds its own bias. Eleven other features do the same job with
different stored numbers. `tanh` bends those results; the last layer mixes
all twelve into one answer.

Keep the **recipe's knobs** separate from the **numbers passing through it**:

| Stored across calls | Computed for this call |
|---|---|
| 12 first-layer weights + 12 biases | 12 hidden values per reading |
| 12 output weights + 1 bias | One prediction per reading |
| **37 parameters**, whatever the batch size | **1,920 hidden values** for 160 readings |

Those hidden values are **activations**, not another 1,920 trainable
parameters. A new input changes activations even if every parameter stays put.
The probe below the class reuses its layers with and without `tanh`; its
printed midpoint gaps isolate the bend without training either version.

## 3 - It runs. Does that mean it knows the correction?

Ask the untouched model for all 160 guesses. Correct shape is only the
entrance ticket; the answers can still miss badly.

Compare each guess with its matching reference, square the miss, then average.
For a pencil example, misses of +1 and -1 cancel if averaged directly; their
squared misses average to 1. **Mean squared error (MSE)** avoids that
cancellation. It is not an accuracy percentage.

The next cell calculates MSE both ways and checks agreement. The plot keeps
the individual locations visible after the loss has compressed them into
one score. **Predict:** Will a shape-correct random model already follow
the reference curve?


In [ ]:
loss_function = nn.MSELoss()
with torch.inference_mode():
    initial_prediction = model(x)
    initial_loss = loss_function(initial_prediction, y)
    # [160, 1] misses -> [160, 1] squared misses -> one average number.
    # We combine the errors only to score them, not to mix input examples.
    squared_misses = (initial_prediction - y).square()
    manual_mse = squared_misses.mean()

print(f"Forward shapes: {tuple(x.shape)} -> {tuple(initial_prediction.shape)}")
print(f"Initial MSE: {initial_loss.item():.6f}")
print(f"Same score worked out by hand: {manual_mse.item():.6f}")
assert initial_prediction.shape == y.shape
assert torch.allclose(initial_loss, manual_mse)

# [160, 1] -> [160]: plotting wants a flat list of horizontal/vertical points.
# Drop only the single-value column; keep one point per example.
plot_readings = x.cpu().squeeze(1).numpy()
plot_reference = y.cpu().squeeze(1).numpy()
plot_initial = initial_prediction.cpu().squeeze(1).numpy()
fig, axis = plt.subplots(figsize=(8, 4))
axis.plot(plot_readings, plot_reference, label="reference", color="#bf5700")
axis.plot(plot_readings, plot_initial, label="before practice", color="#2766a3")
axis.set(xlabel="raw sensor reading", ylabel="calibrated value",
         title="A valid shape is not a useful guess")
axis.legend(frameon=False)
axis.grid(alpha=0.2)
plt.show()
plt.close(fig)

The forward pass made one guess per reading, and both MSE calculations
agreed. Running it again with unchanged parameters would not repair the curve.
We have measured the mistake; nothing has acted on it yet.

## 4 - Find the mistake. Then change the weights.

During a tracked forward pass, PyTorch keeps the calculation route needed
to work backward from loss. `loss.backward()` follows that route and writes
**gradients** beside parameters. A gradient says how a tiny parameter
change would affect this loss. It is feedback, not a replacement weight.

`optimizer.step()` is the operation that changes the parameters. Here
**Adam** also keeps running summaries of earlier gradients. That
**optimizer state** belongs to the updater, not to the model's prediction.
The learning rate `0.02` sets an update scale; it does not mean "subtract
0.02 from every weight."

![Backward leaves the weight knob unchanged and fills gradient notes; optimizer.step turns the knob.](images/01-03-backward-is-not-the-update.png)

> **Visual to add — A knob, a feedback note, and an updater's memory**
> Follow the sensor model's `hidden.weight` through three side-by-side
> workbench panels: before backward; after backward with the same knob
> position and a filled `.grad` note; after Adam's step with a moved knob.
> Give Adam a separate memory pad, labelled "running gradient summaries".
> Use `lr=0.02` as a control label, not an illustrated weight delta.
> Suggested caption: "Backward writes advice; the optimizer turns the knob."

The next cell clears old feedback, guesses, scores, copies a weight table,
calls backward, and only then steps. `detach().clone()` makes the copy an
independent snapshot rather than another view of the moving weight.

**Predict:** Will the copied and live weights first differ after backward
or after the step? Read the gradient norm and weight-change norm separately.
Skipping `zero_grad()` would add this feedback to earlier feedback instead
of starting a fresh update.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
model.train()
optimizer.zero_grad()
prediction = model(x)
loss = loss_function(prediction, y)
weights_before = model.hidden.weight.detach().clone()

loss.backward()
gradient_norm = model.hidden.weight.grad.norm().item()
unchanged_after_backward = torch.equal(weights_before, model.hidden.weight.detach())

optimizer.step()
weight_change = (model.hidden.weight.detach() - weights_before).norm().item()
with torch.inference_mode():
    loss_after_one_step = loss_function(model(x), y).item()

print(f"Gradient size after backward: {gradient_norm:.6f}")
print(f"Weights unchanged after backward: {unchanged_after_backward}")
print(f"Weight change after step: {weight_change:.6f}")
print(f"MSE before/after this step: {loss.item():.6f} -> {loss_after_one_step:.6f}")
assert gradient_norm > 0
assert unchanged_after_backward
assert weight_change > 0

The equality check separates feedback from action: backward left the inspected
weight table unchanged; the optimizer moved it. The two printed norms measure
different things, so don't treat gradient size as the amount Adam moved.

The extra forward pass after `step()` scores the **new** parameters. The old
`loss` still describes the old guesses; changing weights does not rewrite a
previously calculated score.

## 5 - One correction isn't a learned curve

Repeat the same five jobs:

```text
clear old gradients -> guess -> measure -> backward -> step
```

An **epoch** is one pass through the examples. All 160 examples fit in one
batch here, so an epoch contains one update. Part 04 will break that
coincidence: several batches mean several updates per epoch.

The inspected first step already counts. The loop adds 299 more, for
300 total. `history[0]` records the untouched model; `history[k]` records
loss **after** update `k`. That is why it holds 301 entries.

Read the printed before/after scores rather than expecting every step to
improve. **Predict:** Would repeating only the forward call, without backward
and step, produce this learning history?


In [ ]:
TRAINING_EPOCHS = 300
# Epoch 0 is the random model; epoch 1 is the step we just checked.
history = [initial_loss.item(), loss_after_one_step]
model.train()
for epoch in range(1, TRAINING_EPOCHS):
    optimizer.zero_grad()
    prediction = model(x)
    loss = loss_function(prediction, y)
    loss.backward()
    optimizer.step()
    # Score the updated model, so history[k] means AFTER k updates.
    with torch.inference_mode():
        updated_loss = loss_function(model(x), y).item()
    history.append(updated_loss)

print(f"Total passes through the examples: {TRAINING_EPOCHS}")
print(f"MSE before/after practice: {history[0]:.6f} -> {history[-1]:.6f}")
print(f"Measured loss reduction: {history[0] / history[-1]:.1f} times")
assert len(history) == TRAINING_EPOCHS + 1
assert history[-1] < history[0]

## 6 - The score fell. Did the curve improve too?

Two views answer two different questions.

- **The fitted curve:** did the learned guesses move toward the reference?
  We'll keep the untouched guesses on the same chart, so the change is visible.
- **The loss history:** how did the average squared miss change as we practised?
  Point zero is before any update; point 300 is after the final one.

These are still the same examples we trained on. A neat fit here doesn't tell
us how the sensor would behave in new conditions. We haven't built that test.
For now we're checking whether the training machinery did what we asked.

**Predict:** If you only saw the last loss number, could you tell which
part of the curve was still wrong? Look at the curve as well.

In [ ]:
model.eval()
with torch.inference_mode():
    learned_prediction = model(x)
    final_loss = loss_function(learned_prediction, y).item()

# [160, 1] -> [160] again: one predicted height for each horizontal reading.
plot_learned = learned_prediction.cpu().squeeze(1).numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(plot_readings, plot_reference, label="reference", color="#bf5700")
axes[0].plot(plot_readings, plot_initial, label="before practice",
             color="#2766a3", linestyle="--")
axes[0].plot(plot_readings, plot_learned, label="after practice", color="#007f86")
axes[0].set(title="Same examples, different weights",
            xlabel="raw sensor reading", ylabel="calibrated value")
axes[0].legend(frameon=False)
axes[1].plot(range(len(history)), history, color="#007f86")
axes[1].set(title="Measured loss after each update",
            xlabel="completed passes (0 = untrained)", ylabel="MSE")
for axis in axes:
    axis.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(LESSON_DIR / "images" / "01-04-measured-fit-and-loss.png", dpi=140)
plt.show()
plt.close(fig)
print(f"The plotted model's MSE is {final_loss:.6f}.")
assert abs(final_loss - history[-1]) < 1e-8

### Read the measured curve, not a promised score

![Measured before/after sensor curves and loss after each update.](images/01-04-measured-fit-and-loss.png)
*This is a saved training plot. The plotting cell refreshes it from `history`
when you rerun the notebook; its printout reports that run's final MSE.*

The left panel asks where the guesses improved. The right asks how the
average error changed with practice. Neither turns the synthetic training
rows into an independent test.

## 7 - Use the learned guesser without practising

Now supply three readings absent from the training grid: -1.25, 0.25, and 1.50.
The known rule lets us compare each prediction with a reference. All three
are **inside** the trained range, so this checks interpolation, not a new
sensor or an unfamiliar operating range.

| Switch | What it controls | What it does not do |
|---|---|---|
| `model.eval()` | Prediction-time behavior of layers such as dropout | Disable autograd or update weights |
| `torch.inference_mode()` | Skip gradient-tracking work inside the block | Select the model's layer behavior |

This model has no dropout, so changing mode does not change its formula.
Using both switches keeps their jobs explicit for later models.
`new_readings` and `new_predictions` are both `[3, 1]`. The same 37 parameters
serve three rows just as they served 160.

**Predict:** Should these predictions need a backward graph, or change the
inspected weights? The next cell checks both.


In [ ]:
new_readings = torch.tensor([[-1.25], [0.25], [1.50]], device=device)
model.eval()
inference_weights_before = model.hidden.weight.detach().clone()
with torch.inference_mode():
    new_predictions = model(new_readings)
    # We know the made-up rule, so we can also show the reference at these readings.
    new_reference = 0.55 * new_readings**3 - 0.25 * new_readings + 0.10

print("reading -> prediction | reference")
for reading, predicted, reference in zip(
    new_readings.cpu(), new_predictions.cpu(), new_reference.cpu(),
):
    print(f"{reading.item():+5.2f} -> {predicted.item():+6.3f} | {reference.item():+6.3f}")
print(f"Inference output needs gradients: {new_predictions.requires_grad}")
assert new_predictions.shape == (3, 1)
assert not new_predictions.requires_grad
assert torch.equal(inference_weights_before, model.hidden.weight.detach())

Those are new positions **inside** the range we practised on, not a new kind
of sensor. A plausible answer here is useful, but it isn't a broad test on
unseen real-world data.

## 8 - Your turn: give the same model less practice

Change only `YOUR_EPOCHS`. Keep the data, architecture, learning rate, and
starting weights the same. That makes this a check of the practice budget,
not a comparison of two lucky random starts.

**Predict:** After 40 passes, will it be as close to the reference as after 300?
Then try 10 or 100 and compare the measured losses. More practice needn't
improve every individual step, so let the numbers answer.

We reset the seed and create a fresh model and optimizer below. Continuing
with the already-trained model would not test a shorter budget.

In [ ]:
YOUR_EPOCHS = 40
assert YOUR_EPOCHS > 0
torch.manual_seed(7)
trial = TinyRegressor().to(device)
trial_optimizer = torch.optim.Adam(trial.parameters(), lr=0.02)
for _ in range(YOUR_EPOCHS):
    trial_optimizer.zero_grad()
    trial_loss = loss_function(trial(x), y)
    trial_loss.backward()
    trial_optimizer.step()

# Measure after the final update, just as we did for the full run.
with torch.inference_mode():
    trial_final_loss = loss_function(trial(x), y).item()
print(f"Same start, {YOUR_EPOCHS} passes: MSE {trial_final_loss:.6f}")
print(f"Same start, {TRAINING_EPOCHS} passes: MSE {final_loss:.6f}")
if trial_final_loss > final_loss:
    print("The shorter run left more error in this comparison.")
else:
    print("This budget matched or beat the longer run. Check the curve as well.")

## The short version

- One input reading becomes twelve hidden **activations**, then one answer.
  All 160 examples remain separate; the model stores only 37 parameters.
- `tanh` supplies the bend. The midpoint probe compares the same layers
  with and without it, rather than crediting extra layers alone.
- Forward guesses; MSE measures; backward writes gradients; Adam changes
  parameters. The copied-weight check shows where the change happens.
- Loss history comes from actual updates. Use the printed losses and the
  overlaid curves to judge improvement, including any bumps.
- Inference reuses learned parameters without another update. Three new
  in-range readings are a small interpolation check, not a sensor field trial.
- The shorter-budget drill resets the seed, model, and optimizer. Otherwise
  "40 epochs" could quietly mean 40 more epochs after the first 300.

### Carry the jobs, not the sensor data

Later word guessers replace readings with text IDs and one numeric output
with scores for possible words. A suitable loss still checks known answers,
backward still supplies feedback, and an optimizer still changes parameters.
We have not trained that word guesser here.

| Coverage | Where we stopped |
|---|---|
| Built and measured | Synthetic pairs; nonlinear module; manual/API MSE agreement; bend probe; backward/step separation; 300 updates; inference; shorter-budget trial |
| Explained | Parameters versus activations; Adam's separate memory; evaluation mode versus gradient tracking |
| Later | Mini-batches and weight reload in Part 04; validation and selection in Part 05; word objectives in Stage 03; attention in Stage 04 |

Catch mismatched prediction/target shapes, stale gradients, and training
fit reported as generalization before trusting a long run.

**Next:** [Tensors and Autograd](../02-tensors-and-autograd/02-tensors-and-autograd.ipynb)
opens the containers: what belongs to an example, which axes survive an
operation, and which values still connect to a gradient graph? It briefly
adds word positions because a single sensor column cannot expose all those
mistakes.

[Handwritten theory](01-deep-learning-map-theory.md).
